Prepare the Saunders et al. 2018 mouse brain reference for Tangram: QC, normalization and cell type annotation.
- Input: DropViz counts by region, merged and filtered with preprocessing kindly adopted from https://github.com/theislab/scib-reproducibility/blob/main/notebooks/data_preprocessing/mouse_brain/01_collect_mouse_brain_studies.ipynb
- Leiden results differ between machines, so the annotation below only matches the saved `adatas/20220823_saunders_annotated.h5ad`, which 01 uses.

In [ ]:
from pathlib import Path
import scanpy as sc
base = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/FOXF2/Visium")

In [ ]:
adata = sc.read_h5ad(base / "adatas" / "saunders_merged.h5ad")
adata.var_names_make_unique()
adata.obs_names_make_unique()
adata.var["mt"] = adata.var_names.str.startswith("mt-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True, size=0.2, alpha=0.5)

In [ ]:
adata = adata[(adata.obs.n_genes_by_counts < 6000) & (adata.obs.pct_counts_mt < 20)].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["normalized"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.tl.pca(adata, svd_solver="arpack")
sc.pp.neighbors(adata, n_pcs=20)
sc.tl.umap(adata)
adata = adata[adata.obs.region != "CB"].copy()
sc.tl.leiden(adata, resolution=1.5)
adata = adata[adata.obs.leiden != "34"].copy()  # single cell
sc.pl.umap(adata, color=["leiden", "class", "region"], wspace=0.3)

In [ ]:
markers = ["Pdgfra", "Mcam", "Pecam1", "Cldn5", "Flt1", "Vtn", "Kcnj8", "Pdgfrb", "Acta2", "Tagln", "Myh11", "Dcn", "Col1a1", "Col6a1",
           "Mbp", "Enpp2", "Cspg4", "Dcx", "Pifo", "Foxj1", "Rbfox3", "Tubb3", "Aqp4", "Aif1", "Ptprc", "Cd14", "Trem2"]
markers = [g for g in markers if g in adata.var_names]
sc.pl.dotplot(adata, markers, groupby="leiden")

In [ ]:
labels = [
    "Neurons", "Neurons", "Oligos", "Neurons", "Astrocytes", "Neurons", "Endothelial cells", "Neurons", "Neurons", "Neurons",
    "Oligos", "Neurons", "Neurons", "Oligos", "Neurons", "Neurons", "Neurons", "OPCs", "Astrocytes", "Neurons",
    "Oligos", "Neurons", "SMCs", "Microglia/Macrophages", "Pericytes", "Oligos", "Neurogenesis", "Fibroblasts", "Neurons", "Endothelial cells",
    "Ependymal cells", "Oligos", "Neurons", "Neurons",
]
adata.obs["clusters"] = adata.obs["leiden"].map(dict(zip(map(str, range(len(labels))), labels))).astype("category")
sc.pl.umap(adata, color=["clusters", "cell_type"], legend_loc="on data", legend_fontsize=10, legend_fontoutline=2)
sc.pl.dotplot(adata, markers, groupby="clusters")

In [ ]:
sc.tl.rank_genes_groups(adata, "clusters", method="wilcoxon")  # Tangram training genes (01)
adata.write_h5ad(base / "adatas" / "saunders_annotated.h5ad", compression="gzip")